In [13]:
import os
import mne
import numpy as np
import pandas as pd
from tqdm import tqdm
from scipy.signal import hilbert, butter, filtfilt
from scipy.stats import entropy
from sklearn.preprocessing import StandardScaler
from typing import Optional, Iterable, Tuple
import re

In [14]:
def add_annotations_from_csv(
    raw: mne.io.BaseRaw,
    csv_path: str,
    seizure_label_patterns: Optional[Iterable[str]] = ('seiz', 'cpsz', 'sz'),
    min_confidence: float = 0.0,
    append: bool = True,
    merge_tol: float = 1e-3,
    annotate_background: bool = False,
    channel_specific: bool = False
) -> None:
    """
    Read CSV with columns including: channel,start_time,stop_time,label,confidence
    and add annotations to `raw`.

    Behavior (default):
      - Detect rows considered 'seizure' if label matches any pattern in seizure_label_patterns
      - Keep rows with confidence >= min_confidence
      - Merge overlapping/adjacent seizure intervals across channels (tolerance merge_tol)
      - By default append new annotations to raw.annotations (append=True).
      - By default only create 'seizure' annotations (annotate_background=False).
      - If channel_specific=True create descriptions like 'seizure:FP1-F7' for channel-level annotations.

    Parameters
    ----------
    raw : mne.io.BaseRaw
        The loaded raw EEG object.
    csv_path : str
        Path to the CSV file.
    seizure_label_patterns : iterable of str
        Substrings or regex fragments to identify seizure labels (case-insensitive).
    min_confidence : float
        Minimum confidence to keep that row (0.0-1.0). Set >0 to filter low-confidence rows.
    append : bool
        If True, append new annotations to existing raw.annotations. If False, replace.
    merge_tol : float
        Tolerance (seconds) for merging adjacent/nearby intervals.
    annotate_background : bool
        If True, also add 'background' annotations for non-seizure intervals.
    channel_specific : bool
        If True, create separate annotations per channel (description contains channel).
    """
    # read CSV; skip comment lines beginning with #
    df = pd.read_csv(csv_path, comment='#')
    # normalize and validate columns
    required = {'channel', 'start_time', 'stop_time', 'label'}
    if not required.issubset(df.columns):
        raise ValueError(f"CSV must contain columns: {required}. Got: {df.columns.tolist()}")

    # drop rows with NaN times or nonpositive durations
    df = df.dropna(subset=['start_time', 'stop_time'])
    df = df.astype({'start_time': float, 'stop_time': float})
    df = df[df['stop_time'] > df['start_time']]

    # apply confidence filter if confidence column exists
    if 'confidence' in df.columns and min_confidence > 0.0:
        df = df[df['confidence'].astype(float) >= float(min_confidence)]

    # prepare seizure detection regex (case-insensitive)
    pattern = re.compile('|'.join(re.escape(p) for p in seizure_label_patterns), flags=re.I)

    # mark each row as seizure vs background
    df['is_seizure'] = df['label'].astype(str).apply(lambda s: bool(pattern.search(s)))

    onsets = []
    durations = []
    descriptions = []

    if channel_specific:
        # create channel-specific annotations: include channel in description
        for _, row in df.iterrows():
            desc = ('seizure:' + row['channel']) if row['is_seizure'] else ('background:' + row['channel'])
            onsets.append(float(row['start_time']))
            durations.append(float(row['stop_time']) - float(row['start_time']))
            descriptions.append(desc)
    else:
        # merge across channels: create global seizure intervals (union of all seizure rows)
        seiz_rows = df[df['is_seizure']]
        if not seiz_rows.empty:
            intervals = seiz_rows[['start_time', 'stop_time']].values.astype(float)
            # sort by start
            intervals = intervals[np.argsort(intervals[:, 0])]
            # merge overlapping/adjacent intervals using merge_tol
            merged = []
            cur_start, cur_stop = intervals[0]
            for s, e in intervals[1:]:
                if s <= cur_stop + merge_tol:
                    cur_stop = max(cur_stop, e)
                else:
                    merged.append((cur_start, cur_stop))
                    cur_start, cur_stop = s, e
            merged.append((cur_start, cur_stop))

            for s, e in merged:
                onsets.append(s)
                durations.append(e - s)
                descriptions.append('seizure')

        if annotate_background:
            # background = complement of merged seizure over file duration
            total_start = 0.0
            # attempt to set total_stop from csv header 'duration' or from max stop_time
            if 'stop_time' in df.columns:
                total_stop = max(df['stop_time'].astype(float).max(), raw.times[-1] if hasattr(raw, 'times') else max(df['stop_time'].astype(float)))
            else:
                total_stop = raw.times[-1] if hasattr(raw, 'times') else 0.0

            # compute complement of merged intervals
            merged_seiz = [(s, s + d) for s, d in zip(onsets, durations) if d > 0 and descriptions[onsets.index(s)] == 'seizure'] if onsets else []
            # simpler: use merged list saved earlier if present
            # construct background intervals as gaps between merged seizures
            if onsets and descriptions and 'seizure' in descriptions:
                # build sorted merged seizure intervals
                merged_sorted = sorted([(onsets[i], onsets[i] + durations[i]) for i in range(len(onsets)) if descriptions[i] == 'seizure'])
                cursor = total_start
                for s, e in merged_sorted:
                    if s > cursor + merge_tol:
                        onsets.append(cursor)
                        durations.append(s - cursor)
                        descriptions.append('background')
                    cursor = max(cursor, e)
                if cursor < total_stop - merge_tol:
                    onsets.append(cursor)
                    durations.append(total_stop - cursor)
                    descriptions.append('background')
            else:
                # no seizures: one big background
                onsets.append(total_start)
                durations.append(max(0.0, total_stop - total_start))
                descriptions.append('background')

    if not onsets:
        # nothing to annotate
        return

    # sort annotations by onset
    order = np.argsort(onsets)
    onsets = [onsets[i] for i in order]
    durations = [durations[i] for i in order]
    descriptions = [descriptions[i] for i in order]

    new_ann = mne.Annotations(onset=onsets, duration=durations, description=descriptions)

    if append and getattr(raw, 'annotations', None) is not None and len(raw.annotations) > 0:
        # append: concatenate existing annotations and new ones, preserving orig_time
        old = raw.annotations
        # combine arrays
        all_onsets = np.concatenate([old.onset, new_ann.onset])
        all_durations = np.concatenate([old.duration, new_ann.duration])
        all_desc = np.concatenate([old.description, new_ann.description])
        # sort combined
        idx = np.argsort(all_onsets)
        merged_ann = mne.Annotations(onset=all_onsets[idx].tolist(),
                                     duration=all_durations[idx].tolist(),
                                     description=all_desc[idx].tolist(),
                                     orig_time=old.orig_time)
        raw.set_annotations(merged_ann)
    else:
        raw.set_annotations(new_ann)

In [15]:
target_channels = [
    'EEG FP1-REF', 'EEG FP2-REF',
    'EEG F3-REF',  'EEG F4-REF',
    'EEG C3-REF',  'EEG C4-REF',
    'EEG P3-REF',  'EEG P4-REF',
    'EEG O1-REF',  'EEG O2-REF',
    'EEG F7-REF',  'EEG F8-REF',
    'EEG T3-REF',  'EEG T4-REF',
    'EEG T5-REF',  'EEG T6-REF',
    'EEG FZ-REF',  'EEG CZ-REF', 'EEG PZ-REF'
]

import numpy as np
import mne


def harmonize_channels(raw, target_channels):
    mapping = {ch: ch.replace('-LE', '-REF') for ch in raw.ch_names if '-LE' in ch}
    raw.rename_channels(mapping)
    
    # Use MNE's built-in reordering and padding
    # This is more robust than manual padding
    info = mne.create_info(ch_names=target_channels, sfreq=raw.info['sfreq'], ch_types='eeg')
    data = np.zeros((len(target_channels), raw.n_times))
    
    present_channels = [ch for ch in target_channels if ch in raw.ch_names]
    indices_in_raw = mne.pick_channels(raw.ch_names, include=present_channels)
    indices_in_target = mne.pick_channels(target_channels, include=present_channels)
    
    data[indices_in_target] = raw.get_data(picks=indices_in_raw)
    
    return mne.io.RawArray(data, info)


### Preprocess EEG

#### v1

EEG signals were bandpass filtered (0.5–59 Hz), resampled to 250 Hz, standardized, and segmented into overlapping windows for seizure analysis.  
Each window was assigned detection labels (ictal/non-ictal) and prediction labels (preictal/interictal), while excluding ictal and postictal segments from the seizure prediction dataset.

In [ ]:
def preprocess_eeg_v1(raw, file_id,
                   low_freq=0.5, high_freq=59, resample_rate=250,
                   window_sec=2, stride_sec=1,
                   preictal_duration=300,
                   postictal_duration=300):

    raw.filter(low_freq, high_freq, fir_design='firwin', verbose=False)
    raw.resample(resample_rate, verbose=False)

    scaler = StandardScaler()
    data = scaler.fit_transform(raw.get_data().T).T

    # Extract seizure intervals
    seizures = []
    for ann in raw.annotations:
        if "seiz" in ann['description'].lower():
            seizures.append((ann['onset'], ann['onset'] + ann['duration']))

    fs = raw.info['sfreq']
    window_size = int(window_sec * fs)
    stride_size = int(stride_sec * fs)

    X_all = []
    y_detection_all = []
    y_prediction = []
    window_center_times = []
    file_ids_all = []

    for start_idx in range(0, data.shape[1] - window_size + 1, stride_size):
        end_idx = start_idx + window_size

        start_time = start_idx / fs
        center_time = start_time + window_sec / 2

        segment = data[:, start_idx:end_idx]

        X_all.append(segment)
        window_center_times.append(center_time)
        file_ids_all.append(file_id)

        detect_label = 0
        predict_label = 0
        is_ictal = False
        is_postictal = False

        # -----------------------
        # DETECTION LABEL
        # -----------------------
        for sz_start, sz_end in seizures:

            # Ictal
            if (start_time < sz_end) and ((start_time + window_sec) > sz_start):
                detect_label = 1
                is_ictal = True
                break

            # Postictal
            postictal_start = sz_end
            postictal_end = sz_end + postictal_duration
            if (start_time < postictal_end) and ((start_time + window_sec) > postictal_start):
                is_postictal = True
                break

        y_detection_all.append(detect_label)

        # -----------------------
        # PREDICTION LABEL (NEW LOGIC)
        # -----------------------
        if is_ictal or is_postictal:
            predict_label = -1
        else:
            for sz_start, _ in seizures:
                time_to_seizure = sz_start - center_time

                if 0 < time_to_seizure <= preictal_duration:
                    predict_label = 1
                    break

        y_prediction.append(predict_label)

    if not X_all:
        return (np.array([]), np.array([]), np.array([]),
                np.array([]), np.array([]), np.array([]),
                np.array([]))

    X_all = np.stack(X_all)
    y_detection_all = np.array(y_detection_all)
    y_prediction = np.array(y_prediction)
    window_center_times = np.array(window_center_times)
    file_ids_all = np.array(file_ids_all)

    # Remove ictal/postictal for prediction dataset
    prediction_mask = y_prediction != -1

    X_prediction = X_all[prediction_mask]
    y_prediction_final = y_prediction[prediction_mask]
    pred_center_times = window_center_times[prediction_mask]
    pred_file_ids = file_ids_all[prediction_mask]

    return (X_all,
            y_detection_all,
            window_center_times,
            X_prediction,
            y_prediction_final,
            pred_center_times,
            pred_file_ids)

#### v2.1 
- The EEG recordings are first bandpass filtered (0.5–59 Hz), resampled to 250 Hz, normalized using z-score standardization, and segmented into overlapping windows using a sliding-window approach.  
- For seizure detection, each window is compared with annotated seizure intervals. Windows overlapping seizure activity are labeled as 1 (ictal), while all remaining windows are labeled as 0 (non-ictal).  
- For seizure prediction, the algorithm extracts windows occurring approximately 100–120 seconds before seizure onset and labels them as 1 (preictal). In addition, the first 100 seizure windows are extracted and labeled as 0 to create a binary preictal-versus-seizure prediction dataset.  
- Seizures that occur too early in the recording are excluded from prediction processing to ensure sufficient preictal EEG context is available for learning.

In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler


def preprocess_eeg_v2_1(raw,
                   file_id=None,
                   low_freq=0.5,
                   high_freq=59,
                   resample_rate=250,
                   window_sec=2,
                   stride_sec=1,
                   preictal_duration=100,
                   seizure_segments=100):

    # --------------------------------
    # Filtering and Resampling
    # --------------------------------
    raw.filter(low_freq, high_freq, fir_design='firwin', verbose=False)
    raw.resample(resample_rate, verbose=False)

    scaler = StandardScaler()
    data = scaler.fit_transform(raw.get_data().T).T

    # --------------------------------
    # Extract seizures
    # --------------------------------
    seizures = []

    for ann in raw.annotations:

        desc = ann['description'].lower()

        if any(x in desc for x in ['seiz', 'sz', 'ictal']):
            seizures.append((ann['onset'], ann['onset'] + ann['duration']))

    if len(seizures) == 0:
        raise ValueError("No seizures found in annotations")

    # --------------------------------
    # Window parameters
    # --------------------------------
    fs = raw.info['sfreq']

    window_size = int(window_sec * fs)
    stride_size = int(stride_sec * fs)

    X_all = []
    y_detection = []
    window_center_times = []

    # --------------------------------
    # Sliding window segmentation
    # --------------------------------
    for start_idx in range(0, data.shape[1] - window_size + 1, stride_size):

        end_idx = start_idx + window_size

        start_time = start_idx / fs
        center_time = start_time + window_sec / 2

        segment = data[:, start_idx:end_idx]

        X_all.append(segment)
        window_center_times.append(center_time)

        detect_label = 0

        for sz_start, sz_end in seizures:

            if (start_time < sz_end) and ((start_time + window_sec) > sz_start):
                detect_label = 1
                break

        y_detection.append(detect_label)

    X_all = np.stack(X_all)
    y_detection = np.array(y_detection)
    window_center_times = np.array(window_center_times)

    # --------------------------------
    # Prediction dataset near seizures
    # --------------------------------

    X_pred_list = []
    y_pred_list = []
    pred_time_list = []

    MIN_PREICTAL = int(preictal_duration * 0.9)
    MAX_PREICTAL = preictal_duration * 1.2

    for sz_start, sz_end in seizures:

        # require at least 80% of preictal window
        if sz_start < MIN_PREICTAL:
            continue

        preictal_start = max(0, sz_start - MAX_PREICTAL)

        # -------------------------
        # Preictal windows
        # -------------------------
        pre_idx = np.where(
            (window_center_times >= preictal_start) &
            (window_center_times < sz_start)
        )[0]

        # -------------------------
        # Seizure windows
        # -------------------------
        seiz_idx = np.where(
            (window_center_times >= sz_start) &
            (window_center_times < sz_end)
        )[0][:seizure_segments]

        if len(pre_idx) == 0 or len(seiz_idx) == 0:
            continue

        selected_idx = np.concatenate([pre_idx, seiz_idx])

        X_pred_list.append(X_all[selected_idx])

        y_pred_list.append(
            np.concatenate([
                np.ones(len(pre_idx)),   # preictal
                np.zeros(len(seiz_idx))  # seizure
            ])
        )

        pred_time_list.append(window_center_times[selected_idx])

    if len(X_pred_list) == 0:
        raise ValueError("No prediction windows extracted. Check seizure annotations.")

    X_prediction = np.concatenate(X_pred_list)
    y_prediction = np.concatenate(y_pred_list)
    pred_center_times = np.concatenate(pred_time_list)

    return (
        X_all,
        y_detection,
        window_center_times,
        X_prediction,
        y_prediction,
        pred_center_times
    )

#### v2.2
- The EEG recordings are filtered, resampled, normalized, and segmented into overlapping windows using a sliding-window approach for both seizure detection and prediction tasks.  
- For seizure detection, each window is assigned a binary label based on overlap with annotated seizure intervals, where seizure windows are labeled as 1 and non-seizure windows as 0.  
- For seizure prediction, the method dynamically selects the last `min(x, available pre-seizure windows)` before seizure onset as preictal samples, where `x` is the predefined number of preictal segments (e.g., 90 or 100). These preictal windows are labeled as 1.  
- The first fixed number of seizure windows (e.g., 100 segments) are extracted and labeled as 0, forming a balanced preictal-versus-seizure prediction dataset without discarding seizures that occur early in the recording.

In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler

def preprocess_eeg_v2_2(raw,
                   file_id=None,
                   low_freq=0.5,
                   high_freq=59,
                   resample_rate=250,
                   window_sec=2,
                   stride_sec=1,
                   preictal_segments=90,  # Changed to segment count
                   seizure_segments=100):

    # --------------------------------
    # Filtering and Resampling
    # --------------------------------
    raw.filter(low_freq, high_freq, fir_design='firwin', verbose=False)
    raw.resample(resample_rate, verbose=False)

    scaler = StandardScaler()
    data = scaler.fit_transform(raw.get_data().T).T

    # --------------------------------
    # Extract seizures
    # --------------------------------
    seizures = []
    for ann in raw.annotations:
        desc = ann['description'].lower()
        if any(x in desc for x in ['seiz', 'sz', 'ictal']):
            seizures.append((ann['onset'], ann['onset'] + ann['duration']))

    if len(seizures) == 0:
        raise ValueError("No seizures found in annotations")

    # --------------------------------
    # Window parameters
    # --------------------------------
    fs = raw.info['sfreq']
    window_size = int(window_sec * fs)
    stride_size = int(stride_sec * fs)

    X_all = []
    y_detection = []
    window_center_times = []

    # --------------------------------
    # Sliding window segmentation
    # --------------------------------
    for start_idx in range(0, data.shape[1] - window_size + 1, stride_size):
        end_idx = start_idx + window_size
        start_time = start_idx / fs
        center_time = start_time + window_sec / 2

        segment = data[:, start_idx:end_idx]

        X_all.append(segment)
        window_center_times.append(center_time)

        detect_label = 0
        for sz_start, sz_end in seizures:
            if (start_time < sz_end) and ((start_time + window_sec) > sz_start):
                detect_label = 1
                break
        y_detection.append(detect_label)

    X_all = np.stack(X_all)
    y_detection = np.array(y_detection)
    window_center_times = np.array(window_center_times)

    # --------------------------------
    # Prediction dataset near seizures
    # --------------------------------
    X_pred_list = []
    y_pred_list = []
    pred_time_list = []

    for sz_start, sz_end in seizures:
        # -------------------------
        # Preictal windows (Targeting up to N segments)
        # -------------------------
        # Look at all windows ending before the seizure starts
        all_pre_idx = np.where(window_center_times < sz_start)[0]
        
        # Take the LAST 'preictal_segments' available before the onset
        start_from = max(0, len(all_pre_idx) - preictal_segments)
        pre_idx = all_pre_idx[start_from:]

        # -------------------------
        # Seizure windows
        # -------------------------
        # Take the FIRST 'seizure_segments' available during the seizure
        seiz_idx = np.where(
            (window_center_times >= sz_start) & 
            (window_center_times < sz_end)
        )[0][:seizure_segments]

        # Basic check to ensure we have at least some data for both
        if len(pre_idx) == 0 or len(seiz_idx) == 0:
            continue

        selected_idx = np.concatenate([pre_idx, seiz_idx])

        X_pred_list.append(X_all[selected_idx])

        y_pred_list.append(
            np.concatenate([
                np.ones(len(pre_idx)),   # preictal
                np.zeros(len(seiz_idx))  # seizure
            ])
        )

        pred_time_list.append(window_center_times[selected_idx])

    if len(X_pred_list) == 0:
        raise ValueError("No prediction windows extracted. Check seizure annotations.")

    X_prediction = np.concatenate(X_pred_list)
    y_prediction = np.concatenate(y_pred_list)
    pred_center_times = np.concatenate(pred_time_list)

    return (
        X_all,
        y_detection,
        window_center_times,
        X_prediction,
        y_prediction,
        pred_center_times
    )

#### v3
- The EEG recordings are filtered using a clinical frequency range (0.5–40 Hz), notch-filtered to remove line noise, resampled to 250 Hz, and normalized using robust scaling before segmentation into overlapping windows.  
- For seizure prediction, the method defines a preictal target window extending from 5 minutes to 1 minute before seizure onset, while the final 1-minute interval before seizure onset acts as a seizure prediction horizon (SPH) buffer and is excluded from training.  
- Windows occurring during seizures and within the postictal recovery period are discarded to avoid contamination from ictal and recovery-related activity.  
- All remaining non-seizure windows far from seizure onset are labeled as interictal baseline samples, creating a cleaner research-standard dataset focused on distinguishing interictal and preictal brain states.

In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler
import mne

def preprocess_eeg_research_standard_v3(raw, file_id,
                                     low_freq=0.5, high_freq=40, # Standard clinical range
                                     resample_rate=250,
                                     window_sec=2, stride_sec=1,
                                     SOP_start=300, # 5 mins back
                                     SOP_end=60,    # 1 min SPH buffer
                                     postictal_duration=120):
    """
    Research Standard: Focuses on the transition from Interictal to Preictal.
    Includes a 'Baseline' period and a 'Target' period for every file.
    """
    # 1. Standard Clinical Filtering
    raw.filter(low_freq, high_freq, fir_design='firwin', verbose=False)
    raw.notch_filter(np.arange(60, 121, 60), verbose=False) # Remove line noise
    raw.resample(resample_rate, verbose=False)

    data = raw.get_data()
    # Robust scaling: Handles outliers better than StandardScaler for EEG
    data = (data - np.median(data, axis=1, keepdims=True)) / (np.std(data, axis=1, keepdims=True) + 1e-6)

    seizures = []
    for ann in raw.annotations:
        if any(x in ann['description'].lower() for x in ['seiz', 'sz', 'ict']):
            seizures.append((ann['onset'], ann['onset'] + ann['duration']))

    if not seizures:
        return None

    fs = raw.info['sfreq']
    window_size = int(window_sec * fs)
    stride_size = int(stride_sec * fs)
    
    X_list, y_list, time_list = [], [], []

    # Calculate the very start of the first seizure
    first_sz_start = seizures[0][0]

    for start_idx in range(0, data.shape[1] - window_size + 1, stride_size):
        end_idx = start_idx + window_size
        start_time = start_idx / fs
        center_time = start_time + (window_sec / 2)
        
        label = -1 # Default: discard
        
        # LOGIC:
        # 1. Interictal (0): Anything more than 15 minutes before seizure (if exists) 
        #    OR the very beginning of the file.
        # 2. Preictal (1): The SOP window (e.g., 5 mins to 1 min before).
        # 3. Discard: SPH (1 min to 0 min before) and the Seizure itself.

        time_to_seizure = first_sz_start - center_time
        
        # Check if we are inside ANY seizure (to discard)
        is_in_seizure = any(s <= start_time < e for s, e in seizures)
        is_post_ictal = any(e <= start_time < (e + postictal_duration) for s, e in seizures)

        if is_in_seizure or is_post_ictal:
            continue

        if time_to_seizure > (SOP_start + 300): 
            label = 0 # "Pure" Baseline
        elif SOP_end < time_to_seizure <= SOP_start:
            label = 1 # Preictal Target
        elif time_to_seizure > SOP_end:
            # This is the far-away background that isn't quite 'baseline'
            label = 0 

        if label != -1:
            X_list.append(data[:, start_idx:end_idx])
            y_list.append(label)
            time_list.append(center_time)

    if len(X_list) < 10: # Not enough data to be useful
        return None

    return np.array(X_list), np.array(y_list), np.array(time_list), np.full(len(y_list), file_id)

#### v4
- The EEG recordings are filtered, resampled, standardized, and segmented into overlapping windows for seizure detection and prediction analysis.  
- For seizure detection, windows overlapping annotated seizure intervals are labeled as 1 (ictal), while all remaining windows are labeled as 0 (non-ictal).  
- For seizure prediction, the method extracts a strict fixed-duration dataset consisting of approximately 60 seconds of preictal EEG immediately before seizure onset and 60 seconds of seizure/post-onset EEG immediately after seizure onset.  
- Only seizures with sufficient data available on both sides of the onset are retained, ensuring a balanced and temporally consistent prediction dataset with equal numbers of preictal and seizure windows.

In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler


def preprocess_eeg_v4(raw,
                   file_id=None,
                   low_freq=0.5,
                   high_freq=59,
                   resample_rate=250,
                   window_sec=2,
                   stride_sec=1):

    # --------------------------------
    # Filtering and Resampling
    # --------------------------------
    raw.filter(low_freq, high_freq, fir_design='firwin', verbose=False)
    raw.resample(resample_rate, verbose=False)

    scaler = StandardScaler()
    data = scaler.fit_transform(raw.get_data().T).T

    # --------------------------------
    # Extract seizures
    # --------------------------------
    seizures = []

    for ann in raw.annotations:
        desc = ann['description'].lower()

        if any(x in desc for x in ['seiz', 'sz', 'ictal']):
            seizures.append((ann['onset'], ann['onset'] + ann['duration']))

    if len(seizures) == 0:
        raise ValueError("No seizures found in annotations")

    # --------------------------------
    # Window parameters
    # --------------------------------
    fs = raw.info['sfreq']

    window_size = int(window_sec * fs)
    stride_size = int(stride_sec * fs)

    X_all = []
    y_detection = []
    window_center_times = []

    # --------------------------------
    # Sliding window segmentation (Detection)
    # --------------------------------
    for start_idx in range(0, data.shape[1] - window_size + 1, stride_size):

        end_idx = start_idx + window_size

        start_time = start_idx / fs
        center_time = start_time + window_sec / 2

        segment = data[:, start_idx:end_idx]

        X_all.append(segment)
        window_center_times.append(center_time)

        detect_label = 0

        for sz_start, sz_end in seizures:
            if (start_time < sz_end) and ((start_time + window_sec) > sz_start):
                detect_label = 1
                break

        y_detection.append(detect_label)

    X_all = np.stack(X_all)
    y_detection = np.array(y_detection)
    window_center_times = np.array(window_center_times)

    # --------------------------------
    # Prediction dataset (STRICT 60s + 60s)
    # --------------------------------
    X_pred_list = []
    y_pred_list = []
    pred_time_list = []

    PREICTAL_SEC = 53.5
    POSTICTAL_SEC = 53.5

    # Number of windows we EXPECT
    expected_windows = int(PREICTAL_SEC / stride_sec)  # e.g. 60

    total_duration = data.shape[1] / fs

    for sz_start, sz_end in seizures:

        # -------------------------
        # Skip if not enough preictal or postictal data
        # -------------------------
        if sz_start < PREICTAL_SEC:
            continue

        if (sz_start + POSTICTAL_SEC) > total_duration:
            continue

        preictal_start = sz_start - PREICTAL_SEC
        seizure_end_fixed = sz_start + POSTICTAL_SEC

        # -------------------------
        # Get candidate indices
        # -------------------------
        pre_idx = np.where(
            (window_center_times >= preictal_start) &
            (window_center_times < sz_start)
        )[0]

        seiz_idx = np.where(
            (window_center_times >= sz_start) &
            (window_center_times <= seizure_end_fixed)
        )[0]

        # -------------------------
        # STRICT enforcement: must have enough windows
        # -------------------------
        if len(pre_idx) < expected_windows or len(seiz_idx) < expected_windows:
            continue

        # Take EXACT number of windows
        pre_idx = pre_idx[-expected_windows:]     # last 60 sec before seizure
        seiz_idx = seiz_idx[:expected_windows]    # first 60 sec after onset

        # Debug (optional)
        # print(f"Seizure @ {sz_start:.1f}s | pre: {len(pre_idx)} | seiz: {len(seiz_idx)}")

        selected_idx = np.concatenate([pre_idx, seiz_idx])

        X_pred_list.append(X_all[selected_idx])

        y_pred_list.append(
            np.concatenate([
                np.ones(expected_windows),   # preictal
                np.zeros(expected_windows)   # seizure
            ])
        )

        pred_time_list.append(window_center_times[selected_idx])

    if len(X_pred_list) == 0:
        raise ValueError("No valid seizures with full 60s preictal + postictal found.")

    X_prediction = np.concatenate(X_pred_list)
    y_prediction = np.concatenate(y_pred_list)
    pred_center_times = np.concatenate(pred_time_list)

    return (
        X_all,
        y_detection,
        window_center_times,
        X_prediction,
        y_prediction,
        pred_center_times
    )

### Feature Engineering

In [17]:
import numpy as np
from scipy.signal import hilbert, butter, filtfilt
from scipy.stats import entropy
import os
import mne

# --- Helper Functions (Block B - Corrected) ---

def butter_bandpass_filter(data, lowcut, highcut, fs, order=5):
    """Helper function for bandpass filtering."""
    nyq = 0.5 * fs
    low, high = lowcut / nyq, highcut / nyq
    b, a = butter(order, [low, high], btype='band')
    return filtfilt(b, a, data, axis=-1)

# --- Delay Embedding ---
def delay_embedding(signal, delays):
    """
    Generate multi-lag, multi-dimensional delay embedding.
    delays: list of integer delays in samples. e.g., [5, 50, 100]
    """
    max_delay = max(delays)
    T = len(signal)
    if T <= max_delay:
        return np.empty((0, len(delays) + 1))
    
    # Creates columns for [x(t), x(t+d1), x(t+d2), ...]
    embedded = [signal[max_delay:]]
    for d in delays:
        embedded.append(signal[max_delay-d:-d])
        
    return np.column_stack(embedded)

# --- PSR Feature Computation ---
def compute_psr_features(embedded):
    if embedded.shape[0] < 2:
        return [0, 0, 0]
    dists = np.linalg.norm(np.diff(embedded, axis=0), axis=1)
    LL = np.sum(dists)
    hist, _ = np.histogram(dists, bins=50, density=True)
    hist += 1e-12
    LEEnt = -np.sum(hist**2 * np.log(hist**2))
    NEnt = entropy(hist)
    return [LL, LEEnt, NEnt]

# --- Band Definitions ---
BANDS = {
    'Delta': (0.5, 4),
    'Theta': (4, 8),
    'Alpha': (8, 13),
    'Beta': (13, 30),
    'Gamma': (30, 49)
}

# --- Multiband Phase + Power Feature Extraction (Corrected) ---
def extract_multiband_features_with_power(X_windows, fs, delays): # <-- Renamed param to 'delays'
    """
    Extracts BOTH PSR phase features AND band power features.
    """
    num_windows, num_channels, _ = X_windows.shape
    all_band_features = []

    for band_name, (low_freq, high_freq) in BANDS.items():
        
        # 1. Filter the signal for this band
        windows_band = butter_bandpass_filter(X_windows, low_freq, high_freq, fs)
        
        # --- NEW: Calculate Band Power Features ---
        power_features = np.log(np.var(windows_band, axis=2) + 1e-9)
        
        # --- 2. Get instantaneous phase ---
        X_phase = np.angle(hilbert(windows_band, axis=-1))
        
        # --- 3. Extract PSR features from this band's phase signal ---
        psr_features = np.zeros((num_windows, num_channels * 3))
        for win_idx in range(num_windows):
            win_features_ch = []
            for ch_idx in range(num_channels):
                signal = X_phase[win_idx, ch_idx]
                embedded = delay_embedding(signal, delays) # <-- Use 'delays'
                features = compute_psr_features(embedded)
                win_features_ch.extend(features)
            psr_features[win_idx, :] = win_features_ch
        
        # --- 4. Concatenate Phase + Power Features for this band ---
        all_band_features.append(power_features)
        all_band_features.append(psr_features)

    # 5. Concatenate features from all bands
    return np.concatenate(all_band_features, axis=1)

### Extracting Pred features

#### v1 - few bands + few features

In [ ]:
# --- 1. Define the "Bands of Interest" ---
# We'll focus on the bands most cited in research
PREDICTION_BANDS = {
    'Theta': (4, 8),
    'Alpha': (8, 13),
    'Gamma': (30, 49)
}

# --- 2. The New Lean Feature Extractor ---
def extract_lean_predictive_features_v1(X_windows, fs, delays):
    """
    Extracts a lean, hypothesis-driven feature set for prediction.
    Features: 1 Power (Amplitude) + 1 Line Length (Phase)
    Bands: Theta, Alpha, Gamma
    """
    num_windows, num_channels, _ = X_windows.shape
    all_band_features = []

    for band_name, (low_freq, high_freq) in PREDICTION_BANDS.items():
        
        # 1. Filter the signal for this band
        windows_band = butter_bandpass_filter(X_windows, low_freq, high_freq, fs)
        
        # --- Feature 1: Amplitude (Power) ---
        power_features = np.log(np.var(windows_band, axis=2) + 1e-9)
        all_band_features.append(power_features)
        
        # --- Feature 2: Phase (Line Length) ---
        X_phase = np.angle(hilbert(windows_band, axis=-1))
        
        ll_features = np.zeros((num_windows, num_channels))
        for win_idx in range(num_windows):
            for ch_idx in range(num_channels):
                signal = X_phase[win_idx, ch_idx]
                embedded = delay_embedding(signal, delays)
                
                # We only compute the Line Length
                if embedded.shape[0] < 2:
                    ll_features[win_idx, ch_idx] = 0
                else:
                    dists = np.linalg.norm(np.diff(embedded, axis=0), axis=1)
                    ll_features[win_idx, ch_idx] = np.sum(dists)
                    
        all_band_features.append(ll_features)

    # Concatenate all features:
    # [Theta_Power, Theta_LL, Alpha_Power, Alpha_LL, Gamma_Power, Gamma_LL]
    return np.concatenate(all_band_features, axis=1)

#### v2- all bands + all featuers

In [ ]:
# --- 1. Define all 5 clinical bands ---
PREDICTION_BANDS = {
    'Delta': (0.5, 4),
    'Theta': (4, 8),
    'Alpha': (8, 13),
    'Beta': (13, 30),
    'Gamma': (30, 49)
}

# --- 2. The Multi-Feature Extractor ---
def extract_lean_predictive_features_v2(X_windows, fs, delays):
    """
    Features per channel per band:
    1. Log Variance (Power)
    2. Phase Line Length (LL)
    3. Log Energy Entropy (LEEnt)
    4. Normal Entropy (NEnt)
    """
    num_windows, num_channels, _ = X_windows.shape
    all_band_features = []

    for band_name, (low_freq, high_freq) in PREDICTION_BANDS.items():
        
        # 1. Filter the signal for this band
        windows_band = butter_bandpass_filter(X_windows, low_freq, high_freq, fs)
        
        # --- Feature 1: Amplitude (Power) ---
        power_features = np.log(np.var(windows_band, axis=2) + 1e-9)
        all_band_features.append(power_features)
        
        # --- PSR Features: LL, LEEnt, NEnt ---
        # Get Phase via Hilbert
        X_phase = np.angle(hilbert(windows_band, axis=-1))
        
        # Initialize storage for PSR features for this band
        band_ll = np.zeros((num_windows, num_channels))
        band_leent = np.zeros((num_windows, num_channels))
        band_nent = np.zeros((num_windows, num_channels))

        for win_idx in range(num_windows):
            for ch_idx in range(num_channels):
                signal = X_phase[win_idx, ch_idx]
                embedded = delay_embedding(signal, delays)
                
                # Compute the three PSR metrics
                ll, leent, nent = compute_psr_features(embedded)
                
                band_ll[win_idx, ch_idx] = ll
                band_leent[win_idx, ch_idx] = leent
                band_nent[win_idx, ch_idx] = nent
                    
        all_band_features.append(band_ll)
        all_band_features.append(band_leent)
        all_band_features.append(band_nent)

    # Final feature vector per window: 
    # (5 bands) * (4 features) * (num_channels)
    return np.concatenate(all_band_features, axis=1)

#### V3 - using 3 different bands

In [ ]:
# --- 1. Define the "Bands of Interest" ---
# We'll focus on the bands most cited in research
PREDICTION_BANDS = {
    'Alpha': (8, 13),
    'Beta': (13, 30),
    'Gamma': (30, 49)
}

# --- 2. The New Lean Feature Extractor ---
def extract_lean_predictive_features_v3(X_windows, fs, delays):
    """
    Extracts a lean, hypothesis-driven feature set for prediction.
    Features: 1 Power (Amplitude) + 1 Line Length (Phase)
    Bands: Theta, Alpha, Gamma
    """
    num_windows, num_channels, _ = X_windows.shape
    all_band_features = []

    for band_name, (low_freq, high_freq) in PREDICTION_BANDS.items():
        
        # 1. Filter the signal for this band
        windows_band = butter_bandpass_filter(X_windows, low_freq, high_freq, fs)
        
        # --- Feature 1: Amplitude (Power) ---
        power_features = np.log(np.var(windows_band, axis=2) + 1e-9)
        all_band_features.append(power_features)
        
        # --- Feature 2: Phase (Line Length) ---
        X_phase = np.angle(hilbert(windows_band, axis=-1))
        
        ll_features = np.zeros((num_windows, num_channels))
        for win_idx in range(num_windows):
            for ch_idx in range(num_channels):
                signal = X_phase[win_idx, ch_idx]
                embedded = delay_embedding(signal, delays)
                
                # We only compute the Line Length
                if embedded.shape[0] < 2:
                    ll_features[win_idx, ch_idx] = 0
                else:
                    dists = np.linalg.norm(np.diff(embedded, axis=0), axis=1)
                    ll_features[win_idx, ch_idx] = np.sum(dists)
                    
        all_band_features.append(ll_features)

    # Concatenate all features:
    # [Theta_Power, Theta_LL, Alpha_Power, Alpha_LL, Gamma_Power, Gamma_LL]
    return np.concatenate(all_band_features, axis=1)

#### for v1, v2 , v4

In [ ]:
import os
import numpy as np
import mne
import glob
from tqdm import tqdm


def load_prediction_dataset_lean(folder_path, fs=250, preictal_duration=100):

    all_X_pred_features = []
    all_y_pred_labels = []
    all_pred_times = []
    all_file_ids = []
    meta = []

    delays_in_samples = [int(0.020 * fs), int(0.200 * fs)]

    edf_files = glob.glob(os.path.join(folder_path, '**/*.edf'), recursive=True)

    for file_index, edf_path in enumerate(tqdm(edf_files, desc="Processing Files")):

        csv_path = edf_path.replace('.edf', '.csv')
        if not os.path.exists(csv_path):
            continue

        try:

            raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)

            raw_std = harmonize_channels(raw, target_channels)
            add_annotations_from_csv(raw_std, csv_path)

            (_, _, _,
             X_pred_windows,
             y_pred,
             pred_time) = preprocess_eeg_v4(
                raw_std.copy(),
                file_id=file_index,
                resample_rate=fs,
            )
            pred_file_ids = np.full(len(pred_time), file_index)
            if X_pred_windows.size == 0:
                continue

            # require seizure presence
            if np.sum(y_pred == 1) == 0:
                continue

            X_pred_features = extract_lean_predictive_features_v3(
                X_pred_windows, fs, delays_in_samples
            )

            pred_file_ids = np.full(len(pred_time), file_index)

            all_X_pred_features.append(X_pred_features)
            all_y_pred_labels.append(y_pred)
            all_pred_times.append(pred_time)
            all_file_ids.append(pred_file_ids)

            meta.append({
                'file': edf_path,
                'file_id': file_index,
                'n_segments': X_pred_features.shape[0]
            })

        except Exception as e:
            print(f"❌ Error processing {edf_path}: {e}")

    if len(all_X_pred_features) == 0:
        raise ValueError("No valid prediction segments extracted.")

    final_X_pred = np.concatenate(all_X_pred_features, axis=0)
    final_y_pred = np.concatenate(all_y_pred_labels, axis=0)
    final_pred_times = np.concatenate(all_pred_times, axis=0)
    final_file_ids = np.concatenate(all_file_ids, axis=0)

    print("\n✅ Lean Prediction Dataset ready:")
    print(f"Prediction Segments: {final_X_pred.shape[0]}")
    print(f"Features: {final_X_pred.shape[1]}")
    print(f"Unique EEG files: {len(np.unique(final_file_ids))}")

    return final_X_pred, final_y_pred, final_pred_times, final_file_ids, meta

#### for v3 


In [ ]:
# def load_prediction_dataset_lean(folder_path, fs=250):
#     all_X_pred_features = []
#     all_y_pred_labels = []
#     all_pred_times = []
#     all_file_ids = []
#     meta = []

#     delays_in_samples = [int(0.020 * fs), int(0.200 * fs)]
#     edf_files = glob.glob(os.path.join(folder_path, '**/*.edf'), recursive=True)

#     for file_index, edf_path in enumerate(tqdm(edf_files, desc="Processing Files")):
#         csv_path = edf_path.replace('.edf', '.csv')
#         if not os.path.exists(csv_path):
#             continue

#         try:
#             raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)
#             raw_std = harmonize_channels(raw, target_channels)
#             add_annotations_from_csv(raw_std, csv_path)

#             # --- UPDATED CALL TO PREPROCESS_EEG ---
#             processed_data = preprocess_eeg_research_standard_v3(
#                 raw_std, 
#                 file_id=file_index,
#                 resample_rate=fs,
#                 SOP_start=180,  # 3 minutes back
#                 SOP_end=30      # 30 second buffer (SPH)
#             )

#             # Check if function returned None (file too short or no seizures)
#             if processed_data is None:
#                 continue

#             X_pred_windows, y_pred, pred_time, pred_file_ids = processed_data

#             # Require at least one seizure warning window (y=1)
#             if np.sum(y_pred == 1) == 0:
#                 continue

#             # --- EXTRACT FEATURES ---
#             X_pred_features = extract_lean_predictive_features_v3(
#                 X_pred_windows, fs, delays_in_samples
#             )

#             all_X_pred_features.append(X_pred_features)
#             all_y_pred_labels.append(y_pred)
#             all_pred_times.append(pred_time)
#             all_file_ids.append(pred_file_ids)

#             meta.append({
#                 'file': edf_path,
#                 'file_id': file_index,
#                 'n_segments': X_pred_features.shape[0]
#             })

#         except Exception as e:
#             print(f"❌ Error processing {edf_path}: {e}")

#     # Final concatenation logic remains the same
#     if len(all_X_pred_features) == 0:
#         raise ValueError("No valid prediction segments extracted.")

#     return (
#         np.concatenate(all_X_pred_features, axis=0),
#         np.concatenate(all_y_pred_labels, axis=0),
#         np.concatenate(all_pred_times, axis=0),
#         np.concatenate(all_file_ids, axis=0),
#         meta
#     )

### data loading

In [42]:
folder_path = 'subset_data'
output_filename = 'TUH_dataset_lean_prediction_53.5_53.5_lean_diffbands.npz'


final_X_pred, final_y_pred, final_pred_times, final_file_ids, meta = \
    load_prediction_dataset_lean(
        folder_path,
        fs=250
    )

np.savez_compressed(output_filename,
                    X_pred=final_X_pred,
                    y_pred=final_y_pred,
                    final_pred_times=final_pred_times,
                    eeg_id=final_file_ids,
                    meta=meta)

print(f"\n✅ Saved dataset to {output_filename}")

Processing Files:   0%|          | 0/135 [00:00<?, ?it/s]

Creating RawArray with float64 data, n_channels=19, n_times=75250
    Range : 0 ... 75249 =      0.000 ...   300.996 secs
Ready.
❌ Error processing subset_data\aaaaaaac\s001_2002\02_tcp_le\aaaaaaac_s001_t000.edf: No valid seizures with full 60s preictal + postictal found.
Creating RawArray with float64 data, n_channels=19, n_times=59000
    Range : 0 ... 58999 =      0.000 ...   235.996 secs
Ready.


Processing Files:   1%|▏         | 2/135 [00:00<00:10, 12.58it/s]

❌ Error processing subset_data\aaaaaaac\s001_2002\02_tcp_le\aaaaaaac_s001_t001.edf: No valid seizures with full 60s preictal + postictal found.
Creating RawArray with float64 data, n_channels=19, n_times=65500
    Range : 0 ... 65499 =      0.000 ...   261.996 secs
Ready.
❌ Error processing subset_data\aaaaaaac\s002_2002\02_tcp_le\aaaaaaac_s002_t000.edf: No valid seizures with full 60s preictal + postictal found.
Creating RawArray with float64 data, n_channels=19, n_times=301200
    Range : 0 ... 301199 =      0.000 ...   752.997 secs
Ready.


Processing Files:   3%|▎         | 4/135 [00:01<00:39,  3.32it/s]

...

Processing Files:  99%|█████████▉| 134/135 [03:13<00:01,  1.64s/it]

❌ Error processing subset_data\aaaaatvr\s001_2015\01_tcp_ar\aaaaatvr_s001_t004.edf: No valid seizures with full 60s preictal + postictal found.
Creating RawArray with float64 data, n_channels=19, n_times=42240
    Range : 0 ... 42239 =      0.000 ...   164.996 secs
Ready.


Processing Files: 100%|██████████| 135/135 [03:14<00:00,  1.44s/it]

❌ Error processing subset_data\aaaaatvr\s001_2015\01_tcp_ar\aaaaatvr_s001_t013.edf: No valid seizures with full 60s preictal + postictal found.

✅ Lean Prediction Dataset ready:
Prediction Segments: 21518
Features: 114
Unique EEG files: 98



✅ Saved dataset to TUH_dataset_lean_prediction_53.5_53.5_lean_diffbands.npz
